# MNIST Bags with HopfieldPooling

This is a self-contained Colab experiment based on the MNIST-bags example from [Hopfield Networks is All You Need](https://arxiv.org/abs/2008.02217).

The task is binary multiple instance learning: a bag is positive when it contains at least one target digit. The model is:

```text
MNIST images -> CNN feature extractor -> HopfieldPooling -> sigmoid classifier
```

This version generates the bags locally, so it does not need the separate `AttentionDeepMIL` repository required by the historical notebook.

In [ ]:
%pip install -q git+https://github.com/ml-jku/hopfield-layers.git

import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets

from hflayers import HopfieldPooling

seed = 7
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## 1. Build the MNIST-bags dataset

A bag has shape `[bag_size, 1, 28, 28]`. The label is attached to the whole bag, not to each individual image.

In [ ]:
class MNISTBags(Dataset):
    def __init__(self, mnist, target_digit=9, num_bags=200, bag_size=10, seed=0):
        generator = torch.Generator().manual_seed(seed)
        images = mnist.data.float().div(255.0).unsqueeze(1)
        labels = mnist.targets
        target_indices = torch.where(labels == target_digit)[0]
        non_target_indices = torch.where(labels != target_digit)[0]

        bags, targets = [], []
        for bag_index in range(num_bags):
            choices = torch.randint(
                len(non_target_indices), (bag_size,), generator=generator
            )
            selected = non_target_indices[choices].clone()
            is_positive = bag_index % 2 == 0
            if is_positive:
                target_choice = torch.randint(
                    len(target_indices), (1,), generator=generator
                )
                position = torch.randint(bag_size, (1,), generator=generator)
                selected[position] = target_indices[target_choice]
            bags.append(images[selected])
            targets.append(float(is_positive))

        self.bags = torch.stack(bags)
        self.targets = torch.tensor(targets, dtype=torch.float32)

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, index):
        return self.bags[index], self.targets[index]

In [ ]:
train_mnist = datasets.MNIST('/content/data', train=True, download=True)
test_mnist = datasets.MNIST('/content/data', train=False, download=True)

train_set = MNISTBags(train_mnist, target_digit=9, num_bags=200, bag_size=10, seed=1)
test_set = MNISTBags(test_mnist, target_digit=9, num_bags=100, bag_size=10, seed=2)
train_loader = DataLoader(train_set, batch_size=16, shuffle=True)
test_loader = DataLoader(test_set, batch_size=32)

bags, labels = next(iter(train_loader))
print('bags:', tuple(bags.shape))
print('labels:', tuple(labels.shape), labels[:8].tolist())

## 2. CNN followed by HopfieldPooling

The CNN processes each image independently. HopfieldPooling then receives a set of image embeddings with shape `[batch, bag_size, embedding_dim]` and returns one pooled vector per bag.

In [ ]:
class HopfieldBagClassifier(nn.Module):
    def __init__(self, embedding_dim=128):
        super().__init__()
        self.image_features = nn.Sequential(
            nn.Conv2d(1, 20, kernel_size=5),
            nn.ReLU(),
            nn.MaxPool2d(2, stride=2),
            nn.Conv2d(20, 50, kernel_size=5),
            nn.ReLU(),
            nn.MaxPool2d(2, stride=2),
        )
        self.embedding = nn.Sequential(
            nn.Flatten(),
            nn.Linear(50 * 4 * 4, embedding_dim),
            nn.ReLU(),
        )
        self.pooling = HopfieldPooling(
            input_size=embedding_dim,
            hidden_size=32,
            output_size=embedding_dim,
            num_heads=1,
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(embedding_dim, 1),
        )

    def forward(self, bags):
        batch_size, bag_size = bags.shape[:2]
        images = bags.reshape(batch_size * bag_size, 1, 28, 28)
        features = self.embedding(self.image_features(images))
        features = features.reshape(batch_size, bag_size, -1)
        pooled = self.pooling(features)
        return self.classifier(pooled).squeeze(-1)

model = HopfieldBagClassifier().to(device)
print('logits:', tuple(model(bags.to(device)).shape))

## 3. Train and evaluate

The model outputs one logit per bag. `BCEWithLogitsLoss` combines the sigmoid operation and binary cross-entropy loss.

In [ ]:
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4)

def run_epoch(loader, training):
    model.train(training)
    total_loss, total_correct, total_count = 0.0, 0, 0
    for bags, targets in loader:
        bags, targets = bags.to(device), targets.to(device)
        logits = model(bags)
        loss = loss_fn(logits, targets)
        if training:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        predictions = (logits.sigmoid() >= 0.5)
        total_loss += loss.item() * len(targets)
        total_correct += (predictions == targets.bool()).sum().item()
        total_count += len(targets)
    return total_loss / total_count, total_correct / total_count

history = []
for epoch in range(1, 11):
    train_loss, train_acc = run_epoch(train_loader, training=True)
    with torch.no_grad():
        test_loss, test_acc = run_epoch(test_loader, training=False)
    history.append((train_loss, train_acc, test_loss, test_acc))
    print(
        f'epoch {epoch:02d} | train loss {train_loss:.3f} acc {train_acc:.3f} | '
        f'test loss {test_loss:.3f} acc {test_acc:.3f}'
    )

In [ ]:
import matplotlib.pyplot as plt

history = np.asarray(history)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history[:, 0], label='train')
axes[0].plot(history[:, 2], label='test')
axes[0].set_title('BCE loss')
axes[0].legend()
axes[1].plot(history[:, 1], label='train')
axes[1].plot(history[:, 3], label='test')
axes[1].set_title('Bag accuracy')
axes[1].legend()
plt.show()

## What this experiment demonstrates

The CNN turns each image into an embedding. HopfieldPooling learns a query pattern that selects and aggregates useful image embeddings from the bag. The final classifier only sees the pooled bag representation.

This is a runnable demonstration of HopfieldPooling, not a reproduction of Table 2 from the UCI benchmark. The UCI experiment uses a different architecture: a self-normalizing network maps tabular inputs to Hopfield state and stored patterns, and a softmax output performs classification.